# Model Card & Final Evaluation Report

**Week 2, Day 7:** Document the final model and create comprehensive evaluation report.

**Objectives:**
1. Create Model Card following best practices
2. Generate final evaluation report
3. Document limitations and ethical considerations
4. Prepare handoff for Week 3 (API Development)

**Reference:** project_guide.md Week 2 Day 7

## 1. Setup & Load Model

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib
from datetime import datetime

# Scikit-learn
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, roc_auc_score, precision_recall_curve, auc,
    f1_score, precision_score, recall_score
)

# Load model and metadata
models_dir = Path('../models')
model = joblib.load(models_dir / 'fraud_detector_v1.pkl')
with open(models_dir / 'metadata.json', 'r') as f:
    metadata = json.load(f)

# Load test data
processed_dir = Path('../data/processed')
X_train = joblib.load(processed_dir / 'X_train.pkl')
X_test = joblib.load(processed_dir / 'X_test.pkl')
y_train = joblib.load(processed_dir / 'y_train.pkl')
y_test = joblib.load(processed_dir / 'y_test.pkl')

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

print("Model and data loaded successfully")
print(f"\nModel: {metadata['model_name']}")
print(f"Training Date: {metadata['training_date']}")
print(f"Technique: {metadata['technique']}")

## 2. Model Card Documentation

### What is a Model Card?

A **Model Card** is a standardized document that provides:
- Model purpose and intended use
- Performance metrics across different groups
- Limitations and ethical considerations
- Usage guidelines for deployment

**Reference:** [Mitchell et al. (2019) - Model Cards for Model Reporting](https://arxiv.org/abs/1810.03993)

In [ ]:
# Display Model Card
print("="*80)
print("" + " "*15 + "FRAUD DETECTION MODEL CARD v1.0" + " "*15)
print("="*80)

print("\n" + "─"*80)
print("MODEL DETAILS")
print("─"*80)
print(f"Name:           {metadata['model_name']}")
print(f"Version:        1.0")
print(f"Type:           Binary Classification (Fraud vs Legitimate)")
print(f"Algorithm:      XGBoost (eXtreme Gradient Boosting)")
print(f"Training Date:  {metadata['training_date'][:10]}")

print("\n" + "─"*80)
print("INTENDED USE")
print("─"*80)
print("Primary Purpose:")
print("  Real-time fraud detection for digital payment transactions")
print("")
print("Target Users:")
print("  - Payment processing platforms")
print("  - E-commerce merchants")
print("  - Financial institutions")
print("")
print("Use Cases:")
print("  ✓ Real-time transaction scoring")
print("  ✓ Batch fraud analysis")
print("  ✓ Alert generation for suspicious transactions")

print("\n" + "─"*80)
print("OUT OF SCOPE")
print("─"*80)
print("✗ Identity verification (use with biometric systems)")
print("✗ Credit risk assessment (different data requirements)")
print("✗ Money laundering detection (requires network analysis)")

## 3. Performance Metrics Summary

In [ ]:
# Generate predictions using the model's optimal threshold
threshold = metadata['threshold']
y_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_proba > threshold).astype(int)

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

# Compute metrics
roc_auc = roc_auc_score(y_test, y_proba)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
specificity = tn / (tn + fp)

print("\n" + "─"*80)
print("PERFORMANCE METRICS (Test Set)")
print("─"*80)
print(f"\nDecision Threshold:     {threshold:.4f}")
print(f"\nDiscrimination:")
print(f"  ROC-AUC:               {roc_auc:.4f}  ✓ Exceeds 0.95 target")
print(f"\nClassification:")
print(f"  Precision:              {precision:.4f}  ({precision:.2%} of fraud alerts are correct)")
print(f"  Recall (Sensitivity):   {recall:.4f}  ({recall:.2%} of fraud caught)")
print(f"  Specificity:            {specificity:.4f}  ({specificity:.2%} of legit transactions correctly identified)")
print(f"  F1-Score:               {f1:.4f}")

print(f"\nConfusion Matrix (n={len(y_test):,}):")
print(f"                     Predicted")
print(f"                  ┌────────────┬────────────┐")
print(f"              Legit│   TN: {tn:5,}   │  FP: {fp:5,}   │")
print(f"  Actual     ──────┼────────────┼────────────┤")
print(f"             Fraud│   FN:  {fn:3}    │  TP:  {tp:3}    │")
print(f"                  └────────────┴────────────┘")

print(f"\nBusiness Impact:")
print(f"  • Fraud prevented:      {tp} cases")
print(f"  • Fraud missed:         {fn} cases")
print(f"  • False alarms:         {fp} transactions ({fp/len(y_test)*100:.2f}% of all transactions)")
print(f"  • Correct rejections:   {tn:,} transactions")

## 4. KPI Achievement Report

In [ ]:
# KPI Targets
kpi_targets = {
    'ROC-AUC': 0.95,
    'Recall': 0.90,
    'Precision': 0.85
}

actual_metrics = {
    'ROC-AUC': roc_auc,
    'Recall': recall,
    'Precision': precision
}

print("\n" + "─"*80)
print("KPI ACHIEVEMENT REPORT")
print("─"*80)

kpi_results = []
for metric, target in kpi_targets.items():
    actual = actual_metrics[metric]
    status = "✓ PASS" if actual >= target else "✗ FAIL"
    gap = actual - target
    gap_str = f"(+{gap:.2%})" if gap >= 0 else f"({gap:.2%})"
    kpi_results.append({
        'Metric': metric,
        'Target': target,
        'Actual': actual,
        'Status': status
    })
    print(f"\n{metric}:")
    print(f"  Target:    {target:.2f}")
    print(f"  Actual:    {actual:.4f}  {gap_str}")
    print(f"  Status:    {status}")

passed = sum(1 for r in kpi_results if 'PASS' in r['Status'])
print(f"\n{'='*40}")
print(f"Summary: {passed}/3 KPIs Met")
print(f"{'='*40}")

## 5. Model Comparison Summary (All Approaches Tested)

In [ ]:
# Load baseline models for comparison
xgb_baseline = joblib.load(models_dir / 'xgboost_optuna.pkl')

# Create comparison table
results = []

# XGBoost Default (0.5)
y_proba_default = xgb_baseline.predict_proba(X_test)[:, 1]
y_pred_default = (y_proba_default > 0.5).astype(int)
results.append({
    'Model': 'XGBoost (Default 0.5)',
    'Threshold': 0.5,
    'ROC-AUC': roc_auc_score(y_test, y_proba_default),
    'Recall': recall_score(y_test, y_pred_default),
    'Precision': precision_score(y_test, y_pred_default)
})

# Current model (optimized threshold)
results.append({
    'Model': f"XGBoost (Optimized {threshold:.3f})",
    'Threshold': threshold,
    'ROC-AUC': roc_auc,
    'Recall': recall,
    'Precision': precision
})

comparison_df = pd.DataFrame(results)

print("\n" + "─"*80)
print("MODEL COMPARISON: ALL APPROACHES TESTED")
print("─"*80)
print(comparison_df.to_string(index=False))

# Highlight the selected model
print("\n" + "="*50)
print("SELECTED MODEL: XGBoost (Optimized Threshold)")
print("="*50)
print(f"\nRationale: Best balance of recall and precision for production deployment.")

## 6. Feature Importance Analysis

In [ ]:
# Get feature importance
importance = model.feature_importances_
feature_names = X_train.columns.tolist()

# Create DataFrame
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importance
}).sort_values('Importance', ascending=False)

# Top 15 features
top_15 = importance_df.head(15)

print("\n" + "─"*80)
print("TOP 15 MOST IMPORTANT FEATURES")
print("─"*80)
print(top_15.to_string(index=False))

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(top_15)), top_15['Importance'][::-1])
ax.set_yticks(range(len(top_15)))
ax.set_yticklabels(top_15['Feature'][::-1])
ax.set_xlabel('Feature Importance (F-score)')
ax.set_title('Top 15 Features for Fraud Detection')
plt.tight_layout()
plt.savefig('../docs/images/feature_importance.png', dpi=150)
plt.show()

print("\nSaved: docs/images/feature_importance.png")

## 7. ROC and Precision-Recall Curves

In [ ]:
# Compute curves
fpr, tpr, _ = roc_curve(y_test, y_proba)
precisions, recalls, pr_thresholds = precision_recall_curve(y_test, y_proba)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
axes[0].plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC-AUC = {roc_auc:.4f}')
axes[0].plot([0, 1], [0, 1], 'r--', label='Random Classifier')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate (Recall)')
axes[0].set_title('ROC Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# PR Curve
axes[1].plot(recalls, precisions, 'b-', linewidth=2)
axes[1].scatter([recall], [precision], c='red', s=100, zorder=5, label=f'Selected (threshold={threshold:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../docs/images/final_evaluation_curves.png', dpi=150)
plt.show()

print("Saved: docs/images/final_evaluation_curves.png")

## 8. Limitations & Ethical Considerations

In [ ]:
print("\n" + "─"*80)
print("LIMITATIONS")
print("─"*80)

limitations = [
    ("1. Class Imbalance", "Fraud represents only 0.17% of transactions, making it "
     "difficult to achieve both high recall AND high precision simultaneously."),
    
    ("2. False Positive Impact", f"At {recall:.1%} recall, {fp} legitimate transactions "
     f"are flagged as fraud. This may frustrate customers."),
    
    ("3. False Negative Cost", f"{fn} fraud cases are missed. Each missed fraud "
     "could result in financial loss."),
    
    ("4. Data Drift", "The model was trained on historical data. Fraud patterns "
     "evolve over time, requiring periodic retraining."),
    
    ("5. Feature Blindness", "V1-V28 are PCA-transformed features. We cannot "
     "explain exactly which transaction attributes contribute to fraud predictions.")
]

for title, desc in limitations:
    print(f"\n{title}:")
    print(f"  {desc}")

print("\n" + "─"*80)
print("ETHICAL CONSIDERATIONS")
print("─"*80)

ethical = [
    ("Fairness", "The training data may contain biases. Certain demographic "
     "groups or geographic regions may experience higher false positive rates."),
    
    ("Transparency", "Customers should be informed when transactions are "
     "declined and provided a clear explanation."),
    
    ("Appeals Process", "There must be a mechanism for customers to "
     "appeal fraud determinations."),
    
    ("Regulatory Compliance", "Ensure compliance with PCI DSS, GDPR, and "
     "other relevant regulations.")
]

for title, desc in ethical:
    print(f"\n{title}:")
    print(f"  {desc}")

## 9. Deployment Recommendations

In [ ]:
print("\n" + "─"*80)
print("DEPLOYMENT RECOMMENDATIONS")
print("─"*80)

recommendations = [
    ("1. Start with Shadow Mode", "Run the model in parallel with existing "
     "systems for 1-2 weeks. Compare predictions without blocking transactions."),
    
    ("2. Monitor Key Metrics", "Track these daily:")
]

for title, desc in recommendations:
    print(f"\n{title}")
    print(f"  {desc}")

print("\n     Metrics to Monitor:")
metrics_to_monitor = [
    "• Prediction distribution (should match training)",
    "• Fraud rate (should be ~0.17%)",
    "• Alert rate (should be ~0.2%)",
    "• True positive rate (should remain consistent)"
]
for m in metrics_to_monitor:
    print(f"       {m}")

print("\n  3. Set Up Alerts")
alerts = [
    "• If fraud rate > 0.5% (potential attack)",
    "• If prediction latency > 100ms (SLA breach)",
    "• If true positive rate drops > 10% (model drift)",
    "• If feature distribution shifts (PSI > 0.2)"
]
for a in alerts:
    print(f"     {a}")

print("\n  4. Retraining Schedule")
print("     • Initial: Retrain weekly")
print("     • After stability: Retrain monthly")
print("     • Trigger: If drift detected or performance drops > 5%")

print("\n  5. A/B Testing")
print("     • Test new models on 5% of traffic before full rollout")

## 10. Week 2 Summary

In [ ]:
print("\n" + "="*80)
print("                 WEEK 2: FEATURE ENGINEERING & MODEL TRAINING - SUMMARY")
print("="*80)

days_summary = [
    ("Day 1-2", "Feature Engineering", [
        "✓ Extracted hour and day from time_elapsed",
        "✓ Applied cyclic encoding (sin/cos) to hour",
        "✓ Scaled amount feature using StandardScaler",
        "✓ Split data with stratification (80/20)"
    ]),
    ("Day 3", "Baseline Models", [
        "✓ Trained Logistic Regression (ROC-AUC: 0.96)",
        "✓ Trained Random Forest (ROC-AUC: 0.95)",
        "✓ Trained XGBoost (ROC-AUC: 0.98)",
        "✓ XGBoost selected as best baseline"
    ]),
    ("Day 4-5", "XGBoost Tuning", [
        "✓ Performed hyperparameter optimization with Optuna",
        f"✓ Ran 37 trials",
        f"✓ Best CV ROC-AUC: 0.9857",
        f"✓ Test ROC-AUC: 0.9814"
    ]),
    ("Day 6", "Class Imbalance", [
        "✓ Tested threshold tuning",
        "✓ Tested SMOTE oversampling",
        "✓ Tested balanced class weights",
        f"✓ Selected: XGBoost with optimized threshold"
    ]),
    ("Day 7", "Model Card & Evaluation", [
        "✓ Created model card documentation",
        "✓ Generated final evaluation report",
        "✓ Documented limitations and ethical considerations",
        "✓ Prepared deployment recommendations"
    ])
]

for day, topic, tasks in days_summary:
    print(f"\n{day}: {topic}")
    print("─"*60)
    for task in tasks:
        print(f"  {task}")

print("\n" + "="*80)
print("                         FINAL MODEL STATISTICS")
print("="*80)
print(f"\nModel:               XGBoost v1.0")
print(f"Features:            {len(feature_names)} (V1-V28, amount_scaled, hour_sin, hour_cos)")
print(f"Training Samples:    {len(X_train):,}")
print(f"Test Samples:        {len(X_test):,}")
print(f"\nROC-AUC:             {roc_auc:.4f} ✓")
print(f"Recall:              {recall:.4f}")
print(f"Precision:           {precision:.4f}")
print(f"F1-Score:            {f1:.4f}")
print(f"\nKPIs Met:           {passed}/3")

print("\n" + "="*80)
print("                    NEXT WEEK: API DEVELOPMENT & DEPLOYMENT")
print("="*80)
next_week = [
    "• Build FastAPI prediction service",
    "• Create Docker container",
    "• Set up CI/CD pipeline",
    "• Deploy to AWS EC2",
    "• Build Streamlit monitoring dashboard"
]
for item in next_week:
    print(f"  {item}")

print("\n" + "="*80)

## 11. Save Final Report

In [ ]:
# Generate and save final evaluation report as JSON
final_report = {
    "report_date": datetime.now().isoformat(),
    "model_version": "1.0",
    "model_name": "Fraud Detector v1.0",
    "algorithm": "XGBoost",
    "technique": metadata["technique"],
    "decision_threshold": float(threshold),
    "dataset": {
        "train_samples": int(len(X_train)),
        "test_samples": int(len(X_test)),
        "features": int(len(feature_names)),
        "fraud_rate_train": float(y_train.mean()),
        "fraud_rate_test": float(y_test.mean())
    },
    "performance": {
        "roc_auc": float(roc_auc),
        "recall": float(recall),
        "precision": float(precision),
        "f1_score": float(f1),
        "specificity": float(specificity)
    },
    "confusion_matrix": {
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp)
    },
    "kpi_status": {
        "roc_auc_pass": bool(roc_auc >= 0.95),
        "recall_pass": bool(recall >= 0.90),
        "precision_pass": bool(precision >= 0.85),
        "total_passed": passed
    },
    "feature_importance": {
        "top_5_features": importance_df.head(5)[['Feature', 'Importance']].to_dict('records')
    },
    "limitations": [
        "Extreme class imbalance (0.17% fraud rate)",
        "PCA features lack interpretability",
        "Model may drift over time as fraud patterns evolve",
        "False positives may impact customer experience"
    ],
    "deployment_recommendations": [
        "Start with shadow mode testing",
        "Monitor predictions and fraud rates daily",
        "Set up alerts for performance degradation",
        "Retrain weekly initially, then monthly",
        "Use A/B testing for new model versions"
    ]
}

# Save report
with open(models_dir / 'final_evaluation_report.json', 'w') as f:
    json.dump(final_report, f, indent=2)

print("\n" + "="*60)
print("  FINAL EVALUATION REPORT SAVED")
print("="*60)
print(f"\nFile: {models_dir / 'final_evaluation_report.json'}")
print("\nWeek 2 Day 7 Complete! 🎉")
print("="*60)